# Meteorisk — Comparativa de arquitectura: Google Colab

Este notebook reproduce el pipeline de **Meteorisk** (productor + Spark Structured Streaming + entrenamiento + predicción en streaming) en **Google Colab** para poder comparar contra la ejecución local descrita en `README.md`.

**Sustitución de Kafka:** Colab no tiene Docker. Reemplazamos Kafka por la fuente `rate` de Spark Structured Streaming (genera filas a una velocidad configurable) más una función de mapeo que produce eventos meteorológicos sintéticos con la misma distribución que `producer.py` (incluye anomalías con probabilidad ~5 %). La lógica de limpieza, ventana, watermark, entrenamiento y predicción es **idéntica** a la versión local.

Al final del notebook se descargan métricas comparables a las locales: duraciones de jobs, shuffle, GC, número de tareas, throughput observado.

### 📊 Spark UI & Metrics Accessibility

**Important:** Colab does not expose the Spark driver UI directly. Instead, this notebook:
1. **Enables event logging** to capture all metrics programmatically
2. **Uses the Spark REST API** (section 5) to extract jobs, stages, executors, and performance data
3. **Saves metrics as JSON** files that can be compared with local runs

At the end of the notebook, all metrics are **automatically downloaded** as a ZIP file. Compare these JSON values with metrics from your local runs (captured via `http://localhost:18080` History Server).

See [SPARK_UI_GUIDE.md](https://github.com/alexjimenezg/StreamMetoData/blob/main/Meteorisk/SPARK_UI_GUIDE.md) for detailed comparison instructions.

In [ ]:
# Instalación (~30 s en Colab)
!pip install -q pyspark==3.5.0
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--conf spark.ui.showConsoleProgress=false pyspark-shell'

In [ ]:
import time, json, math, random, shutil, os, urllib.request
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import (col, expr, when, window, avg, count,
                                    max as smax, min as smin, sum as ssum,
                                    to_timestamp, current_timestamp, lit, rand, udf)
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                                IntegerType, TimestampType)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, RandomForestClassificationModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

BASE = '/content/meteorisk'
PROCESSED = f'{BASE}/data/processed'
AGGREGATES = f'{BASE}/data/aggregates'
PREDICTIONS = f'{BASE}/data/predictions'
CKPT = f'{BASE}/data/checkpoints'
MODEL = f'{BASE}/models/weather_risk_model'
METRICS = f'{BASE}/data/metrics'

for p in [PROCESSED, AGGREGATES, PREDICTIONS, CKPT, METRICS]:
    shutil.rmtree(p, ignore_errors=True); os.makedirs(p, exist_ok=True)
shutil.rmtree(MODEL, ignore_errors=True)

# Enable event logging for persistent metrics capture
EVENT_LOG_DIR = f'{BASE}/spark_events'
os.makedirs(EVENT_LOG_DIR, exist_ok=True)

spark = (SparkSession.builder
         .appName('MeteoriskColab')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.eventLog.enabled', 'true')
         .config('spark.eventLog.dir', EVENT_LOG_DIR)
         .config('spark.ui.showConsoleProgress', 'true')
         .getOrCreate())
print('Spark version:', spark.version)
print('Default parallelism:', spark.sparkContext.defaultParallelism)
print('Event logging enabled, storing at:', EVENT_LOG_DIR)
print('App UI: driver web UI available during execution')

## 1. Productor sintético (sustituye Kafka)

Usamos la fuente `rate` de Spark + un mapeo determinístico+aleatorio para generar eventos meteorológicos. La velocidad objetivo coincide con la del load test local (5 000 ev/s).

In [ ]:
RATE_EV_S = 5000          # eventos/segundo
ANOMALY_PROB = 0.05

raw = (spark.readStream
       .format('rate')
       .option('rowsPerSecond', RATE_EV_S)
       .load())

# Mapear `value` (long monotónico) + ruido a un evento meteorológico
events = (raw
    .withColumn('seed', (col('value') % 1000000).cast('int'))
    .withColumn('r1', rand())
    .withColumn('r2', rand())
    .withColumn('temperature',
        when(col('r1') < ANOMALY_PROB, 35.0 + col('r2') * 7.0)
        .otherwise(22.0 + (rand() - 0.5) * 12.0))
    .withColumn('humidity', 60.0 + (rand() - 0.5) * 50.0)
    .withColumn('precipitation',
        when(col('r1') > 1 - ANOMALY_PROB, 50.0 + col('r2') * 40.0)
        .otherwise(rand() * 3.0))
    .withColumn('wind_speed',
        when((col('r1') > ANOMALY_PROB) & (col('r1') < 2*ANOMALY_PROB), 60.0 + col('r2') * 30.0)
        .otherwise(12.0 + (rand() - 0.5) * 8.0))
    .withColumn('wind_gusts', col('wind_speed') + rand() * 8.0)
    .withColumn('surface_pressure', 1015.0 + (rand() - 0.5) * 8.0)
    .withColumn('apparent_temperature', col('temperature') + (rand() - 0.5) * 3.0)
    .withColumn('weather_code', (rand() * 65).cast('int'))
    .withColumn('city', lit('Mexico City'))
    .withColumnRenamed('timestamp', 'timestamp'))

clean = events.select('city', 'timestamp', 'temperature', 'humidity',
                       'precipitation', 'wind_speed', 'wind_gusts',
                       'surface_pressure', 'apparent_temperature', 'weather_code')
print('Schema generado:'); clean.printSchema()

## 2. Streaming + Parquet (análogo a `streaming.py`)

In [ ]:
stats = (clean
    .withWatermark('timestamp', '2 minutes')
    .groupBy(window(col('timestamp'), '1 minute'), col('city'))
    .agg(avg('temperature').alias('avg_temperature'),
         smax('temperature').alias('max_temperature'),
         smin('temperature').alias('min_temperature'),
         avg('humidity').alias('avg_humidity'),
         ssum('precipitation').alias('total_precipitation'),
         smax('wind_speed').alias('max_wind_speed'),
         smax('wind_gusts').alias('max_wind_gusts'),
         avg('surface_pressure').alias('avg_surface_pressure'),
         avg('apparent_temperature').alias('avg_apparent_temperature'),
         count('*').alias('event_count'))
    .withColumn('window_start', col('window.start'))
    .withColumn('window_end', col('window.end'))
    .drop('window'))

q1 = (clean.writeStream.format('parquet')
      .outputMode('append').option('path', PROCESSED)
      .option('checkpointLocation', f'{CKPT}/processed')
      .start())

q2 = (stats.writeStream.format('parquet')
      .outputMode('append').option('path', AGGREGATES)
      .option('checkpointLocation', f'{CKPT}/aggregates')
      .start())

print('Streaming queries:', [q.id for q in (q1, q2)])
STREAM_SECONDS = 60
print(f'Ejecutando {STREAM_SECONDS}s ...')
time.sleep(STREAM_SECONDS)
q1.stop(); q2.stop()
print('Stopped. Files written:'); print(' processed:', len(os.listdir(PROCESSED))); print(' aggregates:', len(os.listdir(AGGREGATES)))

## 3. Entrenamiento (análogo a `train_model.py`)

In [ ]:
df = spark.read.parquet(PROCESSED)
total = df.count(); print('rows:', total)

labeled = df.withColumn('risk_label',
    when((col('temperature') > 35) | (col('wind_speed') > 60) | (col('precipitation') > 50), 2)
    .when(((col('temperature') >= 30) & (col('temperature') <= 35)) |
          ((col('precipitation') >= 20) & (col('precipitation') <= 50)) |
          ((col('wind_speed') >= 40) & (col('wind_speed') <= 60)), 1)
    .otherwise(0))

labeled.groupBy('risk_label').count().orderBy('risk_label').show()

FEATS = ['temperature','humidity','precipitation','wind_speed','wind_gusts',
         'surface_pressure','apparent_temperature','weather_code']
asm = VectorAssembler(inputCols=FEATS, outputCol='features')
feat = asm.transform(labeled)
train, test = feat.randomSplit([0.8, 0.2], seed=42)

t0 = time.time()
rf = RandomForestClassifier(labelCol='risk_label', featuresCol='features', numTrees=30, maxDepth=5, seed=42)
model = rf.fit(train)
train_time = time.time() - t0
preds = model.transform(test)
metrics = {}
for m in ['accuracy','weightedPrecision','weightedRecall','f1']:
    metrics[m] = MulticlassClassificationEvaluator(labelCol='risk_label', predictionCol='prediction', metricName=m).evaluate(preds)
metrics['train_time_s'] = train_time
metrics['total_rows'] = total
print('METRICS:', metrics)
model.write().overwrite().save(MODEL)
import csv
with open(f'{METRICS}/model_metrics.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['metric','value'])
    for k, v in metrics.items(): w.writerow([k, v])

## 4. Predicción en streaming (análogo a `predict_stream.py`)

In [ ]:
loaded = RandomForestClassificationModel.load(MODEL)

raw2 = (spark.readStream.format('rate').option('rowsPerSecond', 500).load())
events2 = (raw2
    .withColumn('r1', rand()).withColumn('r2', rand())
    .withColumn('temperature', when(col('r1') < 0.10, 35.0 + col('r2') * 7.0).otherwise(22.0 + (rand() - 0.5) * 12.0))
    .withColumn('humidity', 60.0 + (rand() - 0.5) * 50.0)
    .withColumn('precipitation', when(col('r1') > 0.90, 50.0 + col('r2') * 40.0).otherwise(rand() * 3.0))
    .withColumn('wind_speed', when((col('r1') > 0.10) & (col('r1') < 0.20), 60.0 + col('r2') * 30.0).otherwise(12.0 + (rand() - 0.5) * 8.0))
    .withColumn('wind_gusts', col('wind_speed') + rand() * 8.0)
    .withColumn('surface_pressure', 1015.0 + (rand() - 0.5) * 8.0)
    .withColumn('apparent_temperature', col('temperature') + (rand() - 0.5) * 3.0)
    .withColumn('weather_code', (rand() * 65).cast('int'))
    .withColumn('city', lit('Mexico City')))

asm = VectorAssembler(inputCols=FEATS, outputCol='features')
feat2 = asm.transform(events2)
p = loaded.transform(feat2).withColumn('risk_prediction',
    when(col('prediction')==0.0, 'normal')
    .when(col('prediction')==1.0, 'moderate')
    .when(col('prediction')==2.0, 'critical')
    .otherwise('unknown'))

pq = (p.select('city','timestamp','temperature','humidity','precipitation','wind_speed',
               'wind_gusts','surface_pressure','apparent_temperature','weather_code',
               'prediction','risk_prediction')
        .writeStream.format('parquet').outputMode('append')
        .option('path', PREDICTIONS).option('checkpointLocation', f'{CKPT}/predictions').start())
print('Predicting 30 s...'); time.sleep(30); pq.stop()
import pandas as pd
pdf = pd.read_parquet(PREDICTIONS)
print('predictions written:', len(pdf))
print(pdf['risk_prediction'].value_counts().to_dict())

## 5. Captura de métricas Spark UI (REST + Status Tracker)

En Colab la UI no es accesible por defecto; usamos la API REST local del driver y el `StatusTracker` para obtener información equivalente.

In [ ]:
import urllib.request, json
sc = spark.sparkContext
app_id = sc.applicationId
ui = sc.uiWebUrl  # URL local
print('App ID:', app_id); print('UI URL:', ui)

def fetch(p):
    try:
        return json.load(urllib.request.urlopen(f'{ui}/api/v1/applications/{app_id}/{p}', timeout=5))
    except Exception as e:
        return {'error': str(e)}

jobs = fetch('jobs'); stages = fetch('stages'); execs = fetch('executors'); env = fetch('environment')
print('jobs:', len(jobs) if isinstance(jobs, list) else jobs)
print('stages:', len(stages) if isinstance(stages, list) else stages)
if isinstance(jobs, list):
    from datetime import datetime
    durs = []
    for j in jobs:
        if j.get('submissionTime') and j.get('completionTime'):
            s = datetime.strptime(j['submissionTime'][:23],'%Y-%m-%dT%H:%M:%S.%f')
            c = datetime.strptime(j['completionTime'][:23],'%Y-%m-%dT%H:%M:%S.%f')
            durs.append((c-s).total_seconds())
    if durs:
        print(f'job duration: avg={sum(durs)/len(durs):.2f}s min={min(durs):.2f}s max={max(durs):.2f}s')
if isinstance(stages, list):
    print('total shuffleRead:', sum(s.get('shuffleReadBytes',0) for s in stages))
    print('total shuffleWrite:', sum(s.get('shuffleWriteBytes',0) for s in stages))
    print('total GC ms:', sum(s.get('jvmGcTime',0) for s in stages))
if isinstance(execs, list):
    for e in execs:
        print(f"executor {e['id']}: cores={e['totalCores']} mem={e.get('maxMemory',0)/1e6:.0f}MB tasks={e.get('completedTasks',0)} run={e.get('totalDuration',0)/1000:.2f}s gc={e.get('totalGCTime',0)/1000:.2f}s")
# Hardware info from /proc/cpuinfo
import subprocess
print('--- HOST ---')
print(subprocess.check_output(['bash','-c','cat /proc/cpuinfo | grep "model name" | head -1; free -h | head -2; nproc']).decode())
with open(f'{METRICS}/colab_spark_jobs.json','w') as f: json.dump(jobs, f)
with open(f'{METRICS}/colab_spark_stages.json','w') as f: json.dump(stages, f)
with open(f'{METRICS}/colab_spark_executors.json','w') as f: json.dump(execs, f)
with open(f'{METRICS}/colab_spark_environment.json','w') as f: json.dump(env, f)
print('Metrics saved to', METRICS)

## 6. Descargar resultados para el informe

Comprimir métricas y bajarlas localmente para insertarlas en la tabla comparativa de README § 17.

In [ ]:
import shutil
shutil.make_archive('/content/meteorisk_colab_metrics', 'zip', METRICS)
from google.colab import files
files.download('/content/meteorisk_colab_metrics.zip')